We will compute with pandas only

MA20 (short-term trend)

MA50 (medium-term trend)

MA200 (long-term trend)

Bollinger Bands (Upper, Middle, Lower)

RSI (14-day)

MACD (12, 26, 9)

In [1]:
#Create directory for saving outputs
import os

output_dir = "final_output/technical_indicators"
os.makedirs(output_dir, exist_ok=True)


In [2]:
# load dataset
import pandas as pd

df = pd.read_csv("final_output/psx_final_dataset.csv")
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values(["Ticker", "Date"])

In [3]:
#Define functions for technical indicators
def add_moving_averages(df):
    df["MA20"] = df["Close"].rolling(window=20).mean()
    df["MA50"] = df["Close"].rolling(window=50).mean()
    df["MA200"] = df["Close"].rolling(window=200).mean()
    return df


In [4]:
# bolinger bands
def add_bollinger_bands(df):
    df["BB_Middle"] = df["Close"].rolling(20).mean()
    df["BB_Std"] = df["Close"].rolling(20).std()

    df["BB_Upper"] = df["BB_Middle"] + 2 * df["BB_Std"]
    df["BB_Lower"] = df["BB_Middle"] - 2 * df["BB_Std"]
    return df


In [5]:
#relative strength index
def add_rsi(df, period=14):
    delta = df["Close"].diff()

    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)

    avg_gain = gain.rolling(period).mean()
    avg_loss = loss.rolling(period).mean()

    rs = avg_gain / avg_loss
    df["RSI"] = 100 - (100 / (1 + rs))
    return df


In [6]:
#MACD (Trend Momentum Indicator)
def add_macd(df):
    df["EMA12"] = df["Close"].ewm(span=12, adjust=False).mean()
    df["EMA26"] = df["Close"].ewm(span=26, adjust=False).mean()

    df["MACD"] = df["EMA12"] - df["EMA26"]
    df["MACD_Signal"] = df["MACD"].ewm(span=9, adjust=False).mean()
    return df


In [7]:
#Apply indicators for ALL companies & save per ticker
tickers = df["Ticker"].unique()

for ticker in tickers:
    df_t = df[df["Ticker"] == ticker].copy()

    # Add indicators
    df_t = add_moving_averages(df_t)
    df_t = add_bollinger_bands(df_t)
    df_t = add_rsi(df_t)
    df_t = add_macd(df_t)

    # Save as CSV
    save_path = f"{output_dir}/{ticker}_indicators.csv"
    df_t.to_csv(save_path, index=False)

    print(f"Saved indicators for {ticker}")


Saved indicators for DGKC
Saved indicators for HBL
Saved indicators for LUCK
Saved indicators for MCB
Saved indicators for OGDC
